# Upload results and dataset and functions files

In [ ]:
from google.colab import files
uploaded = files.upload()

# Unzip dataset file and set the correct directory

In [ ]:
import zipfile
import os

zip_file_path = 'dataset_results_functions.zip'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')


os.chdir('/content/')

## Set the dataset and protected attribute

In [ ]:
import numpy as np

nb_seed = 1
dataset = 'adult' # select one in ['adult', 'LSAC']
split_strategy = 'uniform'
mechanism = "OPT"

# read LGBM hyparameters of non-private model
params = np.load('results/' + dataset + '/non_private' + '/LGBM_hyperparameters.npy', allow_pickle='TRUE').item()

# set mechanism folder
folder_name = 'results/' + dataset + "/" + mechanism + "/" +  split_strategy

# for ML
test_size = 0.2 # test proportion for train_test_split
if dataset == 'adult':
    target = 'income'
    protected_attribute = 'race_gender'

elif dataset == 'ACSCoverage':
    target = 'PUBCOV'
    protected_attribute = 'DIS'

elif dataset == 'LSAC':
    target = 'pass_bar'
    protected_attribute = 'fam_inc'

# for privacy
lst_eps = [0.05, 0.25, 0.5, 1, 2, 4, 8, 10, 20] # epsilon-LDP values
if dataset == 'adult':
    #lst_sensitive_att = [protected_attribute, 'race', 'native-country', 'age']
    lst_sensitive_att = [protected_attribute]
elif dataset == 'LSAC':
    #lst_sensitive_att = [protected_attribute, 'fam_inc', 'gender', 'fulltime']
    lst_sensitive_att = [protected_attribute]

#Compute Data Probability Distribution for Optimal LDP mechanism

In [ ]:
import pandas as pd
import numpy as np
import pickle  # For saving and loading data

def compute_probabilities(df, sensitive_attribute, target):
    """
    Computes the following probabilities:
    - p_i: Probability distribution of the sensitive attribute
    - p_1_given_i: Probability of Y=1 given each value of the sensitive attribute
    - Pr_Y_1: Probability of Y=1 in the dataset

    :param df: Pandas DataFrame containing the dataset
    :param sensitive_attribute: The sensitive attribute for which probabilities are computed
    :param target: The target variable (Y)
    :return: p_i, p_1_given_i, Pr_Y_1
    """
    # Compute Pr_Y_1: Probability of Y = 1 in the dataset
    Pr_Y_1 = (df[target] == 1).mean()

    # Get the unique values of the sensitive attribute
    unique_vals = df[sensitive_attribute].unique()

    # Compute p_i: Probability of the sensitive attribute being equal to each value
    p_i = df[sensitive_attribute].value_counts(normalize=True).sort_index()

    # Compute p_1_given_i: Probability of Y = 1 given the sensitive attribute is equal to each value
    p_1_given_i = df.groupby(sensitive_attribute)[target].mean()

    return p_i, p_1_given_i, Pr_Y_1

def save_probabilities(dataset_name, sensitive_attr_probabilities, filepath):
    """
    Saves the computed probabilities for the dataset to a file using pickle.

    :param dataset_name: Name of the dataset (e.g., 'LSAC', 'adult', 'ACSCoverage')
    :param sensitive_attr_probabilities: Dictionary with sensitive attributes as keys and their probabilities as values
    :param filepath: Path where the probabilities will be saved
    """
    with open(filepath, 'wb') as f:
        pickle.dump({dataset_name: sensitive_attr_probabilities}, f)

def load_probabilities(filepath):
    """
    Loads saved probabilities from a file.

    :param filepath: Path to the file containing saved probabilities
    :return: Dictionary of probabilities
    """
    with open(filepath, 'rb') as f:
        return pickle.load(f)


# Load the dataset
if dataset == 'adult':
    df = pd.read_csv('datasets/db_adult_processed_26k.csv')
elif dataset == 'LSAC':
    df = pd.read_csv('datasets/db_LSAC.csv')


# Encode 'race' and 'gender' as categorical variables
df['race'] = df['race'].astype('category').cat.codes
df['gender'] = df['gender'].astype('category').cat.codes

# Merge 'race' and 'gender' into 'race_gender'
df['race_gender'] = df['race'] * df['gender'].nunique() + df['gender']

# Verify the unique values in 'race_gender'
print(df['race_gender'].unique())

# Dictionary to store probabilities for each sensitive attribute
sensitive_attr_probabilities = {}

# Compute and save probabilities for each sensitive attribute
for sensitive_attribute in lst_sensitive_att:
    print(f"Computing probabilities for sensitive attribute: {sensitive_attribute}")
    p_i, p_1_given_i, Pr_Y_1 = compute_probabilities(df, sensitive_attribute, target)

    # Save the results for the current sensitive attribute
    sensitive_attr_probabilities[sensitive_attribute] = {
        "p_i": p_i,
        "p_1_given_i": p_1_given_i,
        "Pr_Y_1": Pr_Y_1
    }

# Save all probabilities to a file
save_filepath = f'{dataset}_probabilities.pkl'  # Save as .pkl file
save_probabilities(dataset, sensitive_attr_probabilities, save_filepath)

# Later, you can load the saved probabilities like this:
loaded_probabilities = load_probabilities(save_filepath)
print(f"Loaded probabilities: {loaded_probabilities}")

#Defining MinMax optimization Problem with data probabilities

run for different epsilon values separately

In [ ]:
import numpy as np


epsilon = 4
# Compute probabilities for the chosen sensitive attribute
p_i, p_1_given_i, Pr_Y_1 = compute_probabilities(df, protected_attribute, target)
k = p_i.shape[0]
# epsilon = [0.05, 0.25, 0.5, 1, 2, 4, 8, 10, 20]
delta = 1  # Example delta


# Helper function to convert matrix indices to 1D vector index
def m2v_idx(i, j, k):
    if i == j:
        raise ValueError("Diagonal elements are not part of the 1D array")

    if i < j:
        return i * (k - 1) + j - 1
    else:
        return i * (k - 1) + j

# k = 4
# print(m2v_idx(1, 2, k))  # Output: index of q_12 in 1D array

def v2m_idx(index, k):
    # Loop through rows to find which row the index corresponds to
    row_start = 0
    for i in range(k):
        row_end = row_start + (k - 1)  # The last index in this row
        if row_start <= index < row_end:
            # Find the column j in this row
            offset = index - row_start
            if offset >= i:
                # If the offset is >= i, it means we're past the diagonal, so adjust j
                j = offset + 1
            else:
                j = offset
            return i, j
        row_start = row_end  # Update the start of the next row

    raise ValueError("Index out of bounds for the flattened array.")


import numpy as np

def construct_row_stochastic_matrix(k, input_vector):
    # Initialize the k x k matrix Q with zeros
    Q = np.zeros((k, k))

    # Fill in the non-diagonal elements from the input_vector
    idx = 0
    for i in range(k):
        row_sum = 0
        for j in range(k):
            if i != j:
                Q[i, j] = input_vector[idx]
                row_sum += input_vector[idx]
                idx += 1
        # Calculate the diagonal element to ensure the row sums to 1
        Q[i, i] = 1 - row_sum

    return Q


# Initialize c and e matrices for the numerator and denominator
c = np.zeros((2 * k, k * (k - 1)))  # 2k fractions
e = np.zeros((2 * k, k * (k - 1)))  # 2k fractions
# Constants in the numerators and denominators
f = np.zeros(2 * k)  # Constant terms for the numerators
g = np.zeros(2 * k)  # Constant terms for the denominators

# Fill in c and e matrices (for each a)
for a in range(k):
    # For the numerators (c)
    for j in range(k):
        if j != a:
            c[a, m2v_idx(j, a, k)] += p_1_given_i[j] * p_i[j]
            c[a, m2v_idx(a, j, k)] -= p_1_given_i[a] * p_i[a]
            c[a, m2v_idx(j, a, k)] -= Pr_Y_1 * p_i[j]
            c[a, m2v_idx(a, j, k)] += Pr_Y_1 * p_i[a]



    # For the denominators (e)
    for j in range(k):
        if j != a:
            e[a, m2v_idx(j, a, k)] += Pr_Y_1 * p_i[j]
            e[a, m2v_idx(a, j, k)] -= Pr_Y_1 * p_i[a]


    # bias term of numerator (f)
    f[a] +=  p_1_given_i[a]*p_i[a]
    f[a] -= Pr_Y_1 * p_i[a]

    # bias term of denominator (g)
    g[a] += Pr_Y_1 * p_i[a]


# Duplicate for the negative terms (due to absolute value in the objective)
for a in range(k):
    c[a + k] = -c[a]
    e[a + k] = e[a]
    f[a + k] = - f[a]
    g[a + k] = g[a]

#########################################################################################
# Constructing matrices A and b

# Initialize A matrix and b vector
num_constraints = k * (k - 1) +  k + k * (k - 1) + k * (k - 1) + 1
A = np.zeros((num_constraints, k * (k - 1)))
b = np.zeros(num_constraints)

constraint_idx = 0  # Keep track of the current constraint row

# 1. Privacy constraints
for i in range(k):
    for j in range(k):
        if i != j:
            q_ij_idx = m2v_idx(i, j, k)
            A[constraint_idx, q_ij_idx] += -np.exp(epsilon)
            for a in range(k):
                if a != j:
                    A[constraint_idx, m2v_idx(j, a, k)] += -1
            b[constraint_idx] = -1
            constraint_idx += 1
#print(constraint_idx)

# 2. Sum constraints: sum(q_ia) <= 1
for i in range(k):
    for a in range(k):
        if a != i:
            A[constraint_idx, m2v_idx(i, a, k)] = 1
    b[constraint_idx] = 1
    constraint_idx += 1
##print(constraint_idx)


# 4. Constraint: 1 - sum(q_ia) >= q_ij --> + q_ij + sum(q_ia) <= 1
for i in range(k):
    for j in range(k):
        if i != j:
            q_ij_idx = m2v_idx(i, j, k)
            A[constraint_idx, q_ij_idx] += 1
            for a in range(k):
                if a != i:
                    A[constraint_idx, m2v_idx(i, a, k)] += 1
            b[constraint_idx] = 1
            constraint_idx += 1
# print(constraint_idx)

# 5. Constraint: 1 - sum(q_ja) >= q_ij -->  q_ij + sum(q_ja) <= 1
for i in range(k):
    for j in range(k):
        if i != j:
            q_ij_idx = m2v_idx(i, j, k)
            A[constraint_idx, q_ij_idx] += 1
            for a in range(k):
                if a != j:
                    A[constraint_idx, m2v_idx(j, a, k)] += 1
            b[constraint_idx] = 1
            constraint_idx += 1
print(constraint_idx)

# 6. Summation constraint: sum( (1 - sum(q_ia)) * p_i ) >= 1 - delta
for i in range(k):
    for a in range(k):
        if a != i:
            A[constraint_idx, m2v_idx(i, a, k)] += p_i[i]
    b[constraint_idx] = delta
print(constraint_idx)


######################################################################################################################


import numpy as np
from scipy.optimize import linprog
import pandas as pd

# Placeholder for t^0_i and t^-0_i for all i
t_minus_0_i = []
t_0_minus_i = []

p = len(f)
n = len(c[0])

# Function to solve LP for each i using Charnes-Cooper transformation for minimization
def solve_lp_min(i):
    # Coefficients for the objective function (for minimization)
    c_obj = np.append(c[i, :], f[i])

    # Coefficients for the constraint
    A_eq = np.append(e[i, :], g[i]).reshape(1, -1)
    b_eq = np.array([1])

    # Inequality constraints Az <= b
    A_ineq = np.hstack([A, -b.reshape(-1, 1)])

    # Bounds for the variables (z_j >= 0 and alpha_i > 0)
    bounds = [(0, None)] * n + [(0, None)]



    # Solve the LP
    result = linprog(c_obj, A_ub=A_ineq, b_ub=np.zeros(len(b)), A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if result.success:
        return result.fun  # Optimal value of the objective function
    else:
        raise ValueError(f"LP did not converge for i={i}")

# Function to solve LP for each i using Charnes-Cooper transformation for maximization
def solve_lp_max(i):
    # Coefficients for the objective function (for maximization, negate the objective)
    c_obj = -np.append(c[i, :], f[i])  # Negate for maximization as linprog minimizes by default

    # Coefficients for the constraint
    A_eq = np.append(e[i, :], g[i]).reshape(1, -1)
    b_eq = np.array([1])

    # Inequality constraints Az <= b
    A_ineq = np.hstack([A, -b.reshape(-1, 1)])

    # Bounds for the variables (z_j >= 0 and alpha_i > 0)
    bounds = [(0, None)] * n + [(0, None)]

    # Solve the LP
    result = linprog(c_obj, A_ub=A_ineq, b_ub=np.zeros(len(b)), A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if result.success:
        return -result.fun  # Return the negated value to get the maximization result
    else:
        raise ValueError(f"LP did not converge for i={i}")

# Solve for each i and store the results for minimization and maximization
for i in range(p):
    #print(f"Solving LP for minimization problem i={i}")
    t_minus_0_i.append(solve_lp_min(i))

    # print(f"Solving LP for maximization problem i={i}")
    t_0_minus_i.append(solve_lp_max(i))

# Output the results
print("t_minus_0_i values:", t_minus_0_i)
print("t_0_minus_i values:", t_0_minus_i)

# Define the rectangle T^0 based on the bounds
T_0 = [(t_minus_0_i[i], t_0_minus_i[i]) for i in range(p)]
print("T^0 rectangle:", T_0)
#########################################################################################

import numpy as np
from scipy.optimize import linprog




# Placeholder lists for l_i and u_i
l = []
u = []

# Function to compute l_i for each i (minimization)
def compute_l_i(i):
    # The objective is to minimize sum(e_ij * y_j) + g_i
    # Here, we only minimize the linear part, as the constant g_i is added afterward
    c_obj = e[i, :]  # Coefficients for the objective function

    # Bounds for the variables y_j >= 0
    bounds = [(0, None)] * n

    # Solve the linear programming problem for minimization
    result = linprog(c=c_obj, A_ub=A, b_ub=b, bounds=bounds, method='highs')

    if result.success:
        return result.fun + g[i]  # Add the constant g_i
    else:
        raise ValueError(f"LP minimization did not converge for i={i}")

# Function to compute u_i for each i (maximization)
def compute_u_i(i):
    # The objective is to maximize sum(e_ij * y_j) + g_i
    # We can minimize the negative of the function to achieve maximization
    c_obj = -e[i, :]  # Negate for maximization

    # Bounds for the variables y_j >= 0
    bounds = [(0, None)] * n

    # Solve the linear programming problem for maximization
    result = linprog(c=c_obj, A_ub=A, b_ub=b, bounds=bounds, method='highs')

    if result.success:
        return -result.fun + g[i]  # Add the constant g_i and negate the result
    else:
        raise ValueError(f"LP maximization did not converge for i={i}")

# Compute l_i and u_i for each i
for i in range(p):
    l.append(compute_l_i(i))
    u.append(compute_u_i(i))

# Output the results
print("l_i values:", l)
print("u_i values:", u)
############################################################################################

import numpy as np

def subdivide_rectangle(T):
    """
    Subdivides the rectangle T into two smaller sub-rectangles based on the longest edge.

    Parameters:
    T : list of tuples    # Rectangle T defined by bounds [(t1_lower, t1_upper), ..., (tp_lower, tp_upper)]

    Returns:
    T1, T2 : tuple of lists    # Two subdivided rectangles (T1 and T2)
    """
    # Compute the length of each interval (upper - lower) for each t_i
    interval_lengths = [upper - lower for (lower, upper) in T]

    # Find rho, the index with the maximum length
    rho = np.argmax(interval_lengths)

    # Compute the midpoint for the rho-th interval
    lower_rho, upper_rho = T[rho]
    midpoint_rho = (lower_rho + upper_rho) / 2

    # Create the two new sub-rectangles
    T1 = T.copy()
    T2 = T.copy()

    # Subdivide the rho-th interval in each rectangle
    T1[rho] = (lower_rho, midpoint_rho)  # First half
    T2[rho] = (midpoint_rho, upper_rho)  # Second half

    return T1, T2
####################################################################################################
def compute_max_fraction(c, f, e, g, y):
    """
    Computes the maximum of the given fractional form:
    max_i { (sum_j c_ij * y_j + f_i) / (sum_j e_ij * y_j + g_i) }, for i = 1, ..., p.

    Parameters:
    c : np.array (p x n)  # Coefficients c_ij for all i,j
    f : np.array (p)      # Vector f_i for all i
    e : np.array (p x n)  # Coefficients e_ij for all i,j
    g : np.array (p)      # Vector g_i for all i
    y : np.array (n)      # Vector y_j for all j (the solution vector y)

    Returns:
    max_value : float     # The maximum value of the fractional expression
    """
    p, n = c.shape  # p: number of terms in the max function, n: number of variables in y

    # Compute the fractional values for each i = 1,...,p
    fractional_values = []
    for i in range(p):
        numerator = np.sum(c[i, :] * y) + f[i]
        denominator = np.sum(e[i, :] * y) + g[i]
        fractional_values.append(numerator / denominator)

    # Find the maximum value among all computed fractions
    max_value = max(fractional_values)

    return max_value

# UB_0 = compute_max_fraction(c,f,e,g,y_0)
#########################################################################################################
import numpy as np
from scipy.optimize import linprog

def solve_lrp_t(A, b, c, e, f, g, T):
    """
    Solves the LRP_T problem as described in the image.

    Parameters:
    A : np.array (m x n)  # Matrix A for inequality constraints (y constraints)
    b : np.array (m)      # Vector b for inequality constraints (y constraints)
    c : np.array (p x n)  # Coefficients c_ij for all i,j
    e : np.array (p x n)  # Coefficients e_ij for all i,j
    f : np.array (p)      # Vector f_i for all i
    g : np.array (p)      # Vector g_i for all i
    T : list of tuples    # Rectangle T defined by bounds [(t1_lower, t1_upper), ..., (tp_lower, tp_upper)]

    Returns:
    result: Optimized value of r and the corresponding y values.
    """
    p, n = c.shape  # p: number of fractional functions, n: number of variables

    # Define the variables (y1, y2, ..., yn, t1, t2, ..., tp, r)
    num_vars = n + p + 1  # n variables for y_j, p for t_i, and 1 for r

    # Objective function: minimize r (which is the last variable)
    c_obj = np.zeros(num_vars)
    c_obj[-1] = 1  # We only want to minimize the last variable, which is r

    # Inequality constraints
    A_ineq = []
    b_ineq = []

    # First constraint: t_i - r <= 0 for all i
    for i in range(p):
        row = np.zeros(num_vars)
        row[n + i] = 1  # t_i coefficient
        row[-1] = -1    # r coefficient
        A_ineq.append(row)
        b_ineq.append(0)

    # Constraints involving the affine functions of y, t with u = 1 and l = 0
    for i in range(p):
        # Upper bound constraints (u_i = 1)
        row_upper_1 = np.zeros(num_vars)
        row_upper_1[:n] = c[i, :]  # Coefficients for y_j
        row_upper_1[n + i] = -1*u[i]     # Coefficient for t_i
        row_upper_1[:n] += -T[i][0] * e[i,:]  # Adjust the e_ij part
        A_ineq.append(row_upper_1)
        b_ineq.append(-f[i]-u[i]*T[i][0]+T[i][0]*g[i] )  # Right-hand side is -f_i-u_i t_lower_i

        row_upper_2 = np.zeros(num_vars)
        row_upper_2[:n] = -c[i, :]  # Coefficients for y_j
        row_upper_2[n + i] = 1*u[i]     # Coefficient for t_i
        row_upper_2[:n] += T[i][1] * e[i,:]  # Adjust the e_ij part
        A_ineq.append(row_upper_2)
        b_ineq.append(+f[i]+u[i]*T[i][1]-T[i][1]*g[i])  # Right-hand side

        # Lower bound constraints (l_i = 0)
        row_lower_1 = np.zeros(num_vars)
        row_lower_1[:n] = c[i, :]  # Coefficients for y_j
        row_lower_1[n + i] = -1*l[i]     # Coefficient for t_i
        row_lower_1[:n] += -T[i][1] * e[i,:]  # Adjust the e_ij part
        A_ineq.append(row_lower_1)
        b_ineq.append(-f[i]-l[i]*T[i][1]+T[i][1]*g[i])  # Right-hand side is -f_i

        row_lower_2 = np.zeros(num_vars)
        row_lower_2[:n] = -c[i, :]  # Coefficients for y_j
        row_lower_2[n + i] = 1*l[i]     # Coefficient for t_i
        row_lower_2[:n] += T[i][0] * e[i,:] # Adjust the e_ij part
        A_ineq.append(row_lower_2)
        b_ineq.append(f[i]+l[i]*T[i][0]-T[i][0]*g[i])  # Right-hand side is f_i

    # Domain constraint: Ay <= b (y \in D)
    for j in range(A.shape[0]):
        row_domain = np.zeros(num_vars)
        row_domain[:n] = A[j, :]  # Apply the A matrix to y_j
        A_ineq.append(row_domain)
        b_ineq.append(b[j])

    # Convert to numpy arrays
    A_ineq = np.array(A_ineq)
    b_ineq = np.array(b_ineq)

    # print(f"A_inequ : \n \n ",A_ineq)
    # print(f"\n \n b_ineq: \n \n", b_ineq)

    # Bounds for the variables (y_j >= 0 and t_i within bounds)
    bounds = [(0, None)] * n + [(T[i][0], T[i][1]) for i in range(p)] + [(None, None)]  # No bound on r

    # Solve the LP
    result = linprog(c_obj, A_ub=A_ineq, b_ub=b_ineq, bounds=bounds, method='highs')
    # print(result.status)
    if result.success:
        return result.x[-1], result.x[:n] # Return optimized r and the y values
    else:
        raise ValueError("LRP_T problem did not converge")


# Call the function to solve LRP_T
LB_0, y_0 = solve_lrp_t(A, b, c, e, f, g, T_0)
print("Optimal Value of LRP_T0:", LB_0)
print("Optimal y of LRP_T0:", y_0)
################################################################################################
def subdivide_and_stop_by_epsilon(A, b, c, e, f, g, T_0, epsilon=1e-5):
    """
    Subdivides rectangles and stops when UB - LB <= epsilon. Prints the iteration when the stopping criterion is met.

    Parameters:
    A : np.array (m x n)  # Matrix A for inequality constraints (y constraints)
    b : np.array (m)      # Vector b for inequality constraints (y constraints)
    c : np.array (p x n)  # Coefficients c_ij for all i,j
    e : np.array (p x n)  # Coefficients e_ij for all i,j
    f : np.array (p)      # Vector f_i for all i
    g : np.array (p)      # Vector g_i for all i
    T_0 : list of tuples  # Initial rectangle defined by bounds [(t1_lower, t1_upper), ..., (tp_lower, tp_upper)]
    epsilon : float       # Stopping tolerance for UB - LB

    Returns:
    y_min_ub : np.array   # The solution y corresponding to the minimum UB
    UB_min : float        # The minimum UB found
    """

    # Initialize with the initial rectangle
    rectangles = [T_0]
    best_y = None
    min_ub = float('inf')  # Start with an infinite UB to find the minimum
    iteration_count = 0  # Initialize iteration count

    while True:
        iteration_count += 1  # Increment iteration count for each loop
        new_rectangles = []
        current_min_lb = float('inf')  # Reset minimum LB for the current iteration

        for T in rectangles:
            # Subdivide each rectangle into two new rectangles
            T1, T2 = subdivide_rectangle(T)

            # Solve LRP for both rectangles and compute their UBs
            try:
                LB_1, y_1 = solve_lrp_t(A, b, c, e, f, g, T1)
                UB_1 = compute_max_fraction(c, f, e, g, y_1)
            except ValueError:
                LB_1, UB_1 = float('inf'), float('inf')  # If LRP doesn't converge, ignore

            try:
                LB_2, y_2 = solve_lrp_t(A, b, c, e, f, g, T2)
                UB_2 = compute_max_fraction(c, f, e, g, y_2)
            except ValueError:
                LB_2, UB_2 = float('inf'), float('inf')  # If LRP doesn't converge, ignore

            # If LB of the rectangles is less than the current min UB, keep them
            if LB_1 < min_ub:
                new_rectangles.append(T1)
                if UB_1 < min_ub:
                    min_ub = UB_1
                    best_y = y_1

            if LB_2 < min_ub:
                new_rectangles.append(T2)
                if UB_2 < min_ub:
                    min_ub = UB_2
                    best_y = y_2

            # Update the minimum LB only for the rectangles in this iteration
            current_min_lb = min(current_min_lb, LB_1, LB_2)

        # Check stopping criterion
        if min_ub - current_min_lb <= epsilon:
            print(f"\n Stopping at iteration: {iteration_count}")
            break

        # Update the list of rectangles with the filtered new ones
        rectangles = new_rectangles

    return best_y, min_ub

# Example usage:
# Assuming A, b, c, e, f, g, and T_0 are already defined as per the earlier provided data.
y_min_ub, UB_min = subdivide_and_stop_by_epsilon(A, b, c, e, f, g, T_0, epsilon=1e-2)

print(f"The solution y corresponding to the minimum UB is: {y_min_ub}")
print(f"The minimum UB found is: {UB_min}")

Q  =  construct_row_stochastic_matrix(k, y_min_ub)
Q_df = pd.DataFrame(Q)

# Save the DataFrame to a CSV file
epsilon_str = f"{epsilon:.2f}".replace('.', '_')
delta_str = f"{delta:.2f}".replace('.', '_')  # Format delta to two decimal places and replace '.' with '_'

# Save the DataFrame to a CSV file with the name including both epsilon and delta
Q_df.to_csv(f'Q_{dataset}_eps_{epsilon_str}_delta_{delta_str}.csv', index=False)


## Writing function

In [ ]:
def write(folder_name, values, mechanism, epsilon):
    #with open(folder_name + "/LGBM_results_"+mechanism+"_eps_"+str(epsilon)+".csv", mode='a', newline='') as scores_file:
    with open(folder_name + "/Appendix_LGBM_results_" + mechanism + "_eps_" + str(epsilon) + ".csv", mode='a', newline='') as scores_file:
        scores_writer = csv.writer(scores_file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        scores_writer.writerow(values)
    scores_file.close()

# Installations

In [ ]:
# Install xxhash
!pip install xxhash

!pip install multi-freq-ldpy


# Install dask[dataframe] to address the warning (optional)
!pip install "dask[dataframe]"

## Importing

In [ ]:
# General imports
import pandas as pd
import matplotlib.pyplot as plt
import time
import copy
import csv

# sklearn imports
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, recall_score

# designed functions
from functions import fairness_metrics

# multi-freq-ldpy import
# from multi_freq_ldpy.pure_frequency_oracles.GRR import GRR_Client
from optimal_ldp_functions import binary_SP_optimal_LDP
from optimal_ldp_functions import non_binary_SP_optimal_LDP
from numba import jit

@jit(nopython=True)
def setting_seed(seed):
    """ Function to set seed for reproducibility.
    Calling numpy.random.seed() from interpreted code will
    seed the NumPy random generator, not the Numba random generator.
    Check: https://numba.readthedocs.io/en/stable/reference/numpysupported.html"""

    np.random.seed(seed)

## Reading dataset

In [ ]:
# if dataset == 'adult':
#     df = pd.read_csv('datasets/db_adult_processed_26k.csv')

# elif dataset == 'ACSCoverage':
#     df = pd.read_csv('datasets/db_ACSCoverage.csv')

# elif dataset == 'LSAC':
#     df = pd.read_csv('datasets/db_LSAC.csv')

df.drop(columns = ['race','gender'], inplace=True)
df.head()

## Run LGBM on DP data

In [ ]:
def load_Q_matrices(epsilon, delta, dataset):
    """
    Load the corresponding Q matrix based on epsilon, delta, and dataset.
    Maps specific epsilon and delta values to the respective filenames.

    :param epsilon: Current epsilon value (e.g., 0.05, 50)
    :param delta: Current delta value (e.g., 1.0)
    :param dataset: Name of the dataset (used in the filename)
    :return: Loaded Q matrix
    """

    # Define the formatted strings for epsilon and delta
    epsilon_str = f"{epsilon:.2f}".replace('.', '_')  # Format epsilon as x_y (e.g., 0.05 -> 0_05)
    delta_str = f"{delta:.2f}".replace('.', '_')      # Format delta as x_y (e.g., 1.0 -> 1_00)

    # Generate file name including both epsilon and delta (no path needed, same directory)
    file_name = f"Q_{dataset}_eps_{epsilon_str}_delta_{delta_str}.csv"

    # Directly read the file, skipping the first row (which contains the index or labels)
    try:
        Q_matrix = pd.read_csv(file_name, header=None, index_col=False, skiprows=1).to_numpy()  # Skip the first row
    except FileNotFoundError:
        raise FileNotFoundError(f"Could not find the file {file_name}. Please check the directory and filename.")

    return Q_matrix

# Main code with loop over deltas
header = ["seed", "acc", "f1", "auc", "recall", "SPD", "EOD", "OAD"]

starttime = time.time()
delta = 1
# Domain size of sensitive attributes
lst_k = {att: len(set(df[att])) for att in lst_sensitive_att}
lst_eps = [0.05,0.25,0.5,1,2,8]
for epsilon in lst_eps:
    for delta in [delta]:  # Iterate over delta values as well
        print(f"Running for epsilon: {epsilon}, delta: {delta}")
        os.makedirs(folder_name, exist_ok=True)

        # Write the header of the CSV file
        write(folder_name, header, mechanism, epsilon)

        # Load Q matrix for non-binary sensitive attributes if necessary
        if any(k > 2 for k in lst_k.values()):
            Q_matrix = load_Q_matrices(epsilon, delta, dataset)
            print(Q_matrix)
            print(f"Loaded Q_matrix shape: {Q_matrix.shape} for epsilon: {epsilon} and delta: {delta}")

        for seed in range(nb_seed):
            setting_seed(seed)  # For reproducibility

            # Use original dataset
            X = copy.deepcopy(df.drop(target, axis=1))
            y = copy.deepcopy(df[target])

            # Train test splitting
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, shuffle=True, stratify=y, random_state=seed)
            y_train.reset_index(inplace=True, drop=True)
            y_test.reset_index(inplace=True, drop=True)

            # One-Hot-Encoding + LDP randomization
            lst_df_train = []
            lst_df_test = []
            for col in X_train.columns:
                lst_col_name = [f"{col}_{val}" for val in range(len(set(df[col])))]
                k = len(set(df[col]))
                OHE = np.eye(k)

                if col in lst_sensitive_att:  # LDP randomization
                    majority_group = X_train[col].mode()[0]
                    eps_att = epsilon / len(lst_sensitive_att) if split_strategy == 'uniform' else epsilon * k / sum(lst_k.values())

                    if k == 2:
                        # Apply binary LDP mechanism
                        df_ohe = pd.DataFrame([OHE[binary_SP_optimal_LDP(val, majority_group, eps_att)] for val in X_train[col]], columns=lst_col_name)
                    else:
                        # Apply non-binary LDP mechanism with corresponding Q matrix
                        df_ohe = pd.DataFrame([OHE[non_binary_SP_optimal_LDP(val, k, Q_matrix)] for val in X_train[col]], columns=lst_col_name)

                else:  # Just One-Hot-Encoding
                    df_ohe = pd.DataFrame([OHE[val] for val in X_train[col]], columns=lst_col_name)

                lst_df_train.append(df_ohe)

                # Test set is original, i.e., just one-hot-encoding
                df_ohe_test = pd.DataFrame([OHE[val] for val in X_test[col]], columns=lst_col_name)
                lst_df_test.append(df_ohe_test)

            # Concatenate one-hot-encoded train/test sets
            X_train = pd.concat(lst_df_train, axis=1)
            X_test = pd.concat(lst_df_test, axis=1)

            # Instantiate and train the model
            model = LGBMClassifier(random_state=seed, n_jobs=2, objective="binary")
            model.set_params(**params)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # Performance metrics
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred)

            # Prepare dataset for fairness analysis
            df_fm = pd.concat([X_test, y_test], axis=1)
            df_fm['y_pred'] = y_pred

            # Fairness metrics
            fair_met = fairness_metrics(df_fm, protected_attribute, target)

            # Write results to CSV
            write(folder_name, [
                str(seed), acc, f1, auc, recall, fair_met["SPD"], fair_met["EOD"], fair_met["OAD"]
            ], mechanism, epsilon)

print(f'That took {time.time() - starttime} seconds')


# Save results

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")